### Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Đọc & xử lý dữ liệu

In [3]:
# Đường dẫn đến thư mục dữ liệu
data_path = '/content/drive/MyDrive/IS353 - Mạng xã hội - Nhóm 6/Đồ án môn học/Project/Data/Data_clean/'

# Kiểm tra dữ liệu file 02.diem và diem_Thu

Do 2 bảng dữ liệu này có các cột và thành phần khá giống nhau nên sẽ kiểm tra 2 bảng dữ liệu này có trùng nhau với nhau

In [4]:
df1 = pd.read_excel(data_path + "/01.sinhvien.xlsx")

In [5]:
df2 = pd.read_excel(data_path + "/02.diem.xlsx")
df3 = pd.read_excel(data_path + "/diem_Thu.xlsx")

In [6]:
df2.columns

Index(['mssv', 'mamh', 'sotc', 'namhoc', 'hocky', 'diem', 'trangthai'], dtype='object')

In [7]:
df3.columns

Index(['mssv', 'mamh', 'sotc', 'hocky', 'namhoc', 'diem_hp', 'trangthai'], dtype='object')

In [8]:
print(f'Số lượng mssv có trong diem: {len(df2["mssv"].unique())}')
print(f'Số lượng mssv có trong diem_Thu: {len(df3["mssv"].unique())}')

Số lượng mssv có trong diem: 4050
Số lượng mssv có trong diem_Thu: 17881


In [9]:
def Check_mssv(df1, df2):
  """
    Hàm kiểm tra giá trị mssv của df2 có trong mssv của df1.

    Parameters:
    - df1 (DataFrame): DataFrame.
    - df2 (DataFrame): DataFrame.

    Returns:
    - Danh sách các giá trị của df2 có trong df1.
    """
  # Lấy danh sách các MSSV duy nhất từ cả hai DataFrame
  mssv1 = df1["mssv"].unique()
  mssv2 = df2["mssv"].unique()

  # Tìm các giá trị trong mssv2 có trong mssv1
  mssv_in_both = df2['mssv'].isin(mssv1)

  return df2[mssv_in_both]

def Check_mssv(df1, df2):
    """
    Hàm kiểm tra giá trị mssv của df2 có trong mssv của df1.

    Parameters:
    - df1 (DataFrame): DataFrame.
    - df2 (DataFrame): DataFrame.

    Returns:
    - Danh sách các giá trị của df2 có trong df1.
    """
    # Lấy danh sách MSSV từ df1 và df2
    mssv_df1 = set(df1['mssv'])
    mssv_df2 = set(df2['mssv'])

    # mssv có trong cả hai bảng
    mssv_chung = mssv_df1.intersection(mssv_df2)
    so_luong_chung = len(mssv_chung)

    # MSSV không có trong cả hai bảng
    mssv_chi_df1 = mssv_df1 - mssv_df2
    mssv_chi_df2 = mssv_df2 - mssv_df1
    so_luong_khong_chung = len(mssv_chi_df1) + len(mssv_chi_df2)

    # Kết quả thống kê
    return {
        'so_luong_chung': so_luong_chung,
        'so_luong_khong_chung': so_luong_khong_chung
    }

In [10]:
# kiểm tra giá trị mssv của df2 có trong mssv của df3
mssv2_3 = Check_mssv(df2, df3)
mssv2_3

{'so_luong_chung': 4050, 'so_luong_khong_chung': 13831}

In [11]:
len(df2['mssv'].unique())

4050

In [12]:
len(df3['mssv'].unique())

17881

In [13]:
df2.shape

(98963, 7)

In [14]:
df3.shape

(674273, 7)

In [15]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df2, df3, on=["mssv", "mamh", "sotc","hocky", "namhoc"], how="left")

# Thêm cột mới để kiểm tra `diemhp` có bằng `diem` không
merged_df["is_equal"] = merged_df["diem_hp"] == merged_df["diem"]

# Xem kết quả
print(merged_df["is_equal"].value_counts())

is_equal
True     98924
False       39
Name: count, dtype: int64


In [16]:
# Kiểm tra khuyết
merged_df.isnull().sum()

,0
mssv,0
mamh,0
sotc,0
namhoc,0
hocky,0
diem,0
trangthai_x,0
diem_hp,1
trangthai_y,1
is_equal,0


In [17]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df2, df3, on=["mssv", "mamh", "sotc", "hocky", "namhoc"], how="right")

# Thêm cột mới để kiểm tra `diemhp` có bằng `diem` không
merged_df["is_equal"] = merged_df["diem_hp"] == merged_df["diem"]

# Xem kết quả
print(merged_df["is_equal"].value_counts())

is_equal
False    575349
True      98924
Name: count, dtype: int64


In [18]:
# Kiểm tra khuyết
merged_df.isnull().sum()

,0
mssv,0
mamh,0
sotc,0
namhoc,0
hocky,0
diem,575311
trangthai_x,575311
diem_hp,0
trangthai_y,0
is_equal,0


Dựa vào kết quả gộp theo df2 hay df3 thì ta thấy được rằng df3 và df2 có dữ liệu trùng nhau và df3 có lượng dữ liệu lớn hơn df2 do đó việc chọn 1 trong 2 bảng sẽ phụ thuộc vào số lượng dữ liệu của df2 hoặc df3 có trong df1 lớn hơn

In [19]:
# kiểm tra giá trị mssv của df2 có trong mssv của df1
mssv1_2 = Check_mssv(df1, df2)
mssv1_2

{'so_luong_chung': 4050, 'so_luong_khong_chung': 4245}

In [20]:
# kiểm tra giá trị mssv của df3 có trong mssv của df1
mssv1_3 = Check_mssv(df1, df3)
mssv1_3

{'so_luong_chung': 8273, 'so_luong_khong_chung': 9630}

Dựa vào kết quả trên ta sẽ chọn df3 tức là diem_Thu để lấy được lượng dữ liệu lớn nhất

# Kiểm tra dữ liệu file 04.xeploaiav

In [21]:
df4 = pd.read_excel(data_path + "/04.xeploaiav.xlsx")

In [22]:
mssv1_4 = Check_mssv(df1, df4)
mssv1_4

{'so_luong_chung': 4919, 'so_luong_khong_chung': 4800}

In [23]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df4, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,diemta,mamta
4334,E64CBFE4XPvAibaEXe/AdnwkoyhfDpgHynAQnBAo,1999,0,bàrịa-vũngtàu,htcl2017.1,httt,clc,12,42.0,eng01
7005,AE3AFF71XPvAibaEXe+boSxJoV2lkOgHbv83iTMq,2001,1,hồ chí minh,htcl2019.1,httt,clc,14,285.0,toeic
2405,8C4FE4F4XPvAibaEXe8Lp0YaxzdqUjpNyxA6voae,1996,1,bìnhphước,cntt0001,kttt,cqui,10,NaN,NaN
156,85FA5549XPvAibaEXe9X5bpEKoa0msFO9rQKADMb,1995,0,bìnhdương,mmtt2013,mmt&tt,cqui,8,NaN,NaN
3633,EAB575CAXPvAibaEXe/A1zCqCr2yayNVcL0naH6W,1998,1,tiềngiang,tmđt2016,httt,cqui,11,58.0,en002


In [24]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
diemta,3376
mamta,3376


Dữ liệu trong file 04.xeploaiav thể hiện là các sinh viên có điểm tiếng anh hay không. Vì vậy với những mssv không có trong file thì sẽ không có bằng hoặc điểm trong môn tiếng anh và mamta sẽ được điền là không và điểm sẽ được điền là 0.

# Kiểm tra dữ liệu file 05.ThiSinh

In [25]:
df5= pd.read_excel(data_path + "05.ThiSinh.xlsx")

In [26]:
mssv1_5 = Check_mssv(df1, df5)
mssv1_5

{'so_luong_chung': 8234, 'so_luong_khong_chung': 61}

In [27]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df5, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,dien_tt,diem_tt,TEN_TRUONG
4249,7E1C7713XPvAibaEXe90TjJq9LqiZnDBhKzMxQ18,1999,1,quảngngãi,khmt2017,khmt,cqui,12,THPT,25.75,trườngthptbìnhsơn
3791,9B423392XPvAibaEXe8G+DN2/i3pr9jcYRxSvWUz,1998,1,đồngnai,ktmt0001,ktmt,cqui,11,THPT,21.75,thptnguyễntrãi
6254,A3126134XPvAibaEXe8jwcgah0IjmiO59OWrX7rw,2000,1,hàtĩnh,mmcl2018.2,mmt&tt,clc,13,THPT,20.10,thptchuyênhàtĩnh
904,9D06D02BXPvAibaEXe/Hpk/kOf6+1d0LuggLW88j,1993,1,quảngngãi,khmt2013,khmt,cqui,8,THPT,25.50,trungtâmgdnn-gdtxhuyệnmộđức
1811,6797E6AFXPvAibaEXe88tz7NL7l/gigtWlWELlnM,1996,1,hồchíminh,antt2014,mmt&tt,cqui,9,THPT,29.00,Không điền


In [28]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
dien_tt,61
diem_tt,61


Dữ liệu trong file 05.ThiSinh thể hiện là các sinh viên không có thông tin trong file và việc này không mang ý nghĩa gì. Vì vậy với những mssv không có trong file thì được điền là không điền theo mode.

# Kiểm tra dữ liệu file 08.XLHV.xlsx

In [29]:
df6 = pd.read_excel(data_path + "08.XLHV.xlsx")

In [30]:
mssv1_6 = Check_mssv(df1, df6)
mssv1_6

{'so_luong_chung': 1863, 'so_luong_khong_chung': 6432}

In [31]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df6, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,tinhtrang,lydo,hocky,namhoc,ngayqd
8166,607684B1XPvAibaEXe+VfAA12Tmd+02WCtZl8NOt,2001,1,vĩnhlong,attt2019,mmt&tt,cqui,14,NaN,NaN,NaN,NaN,NaN
7311,2F86BBD1XPvAibaEXe8EWfnZTwjAD+LMJdMLkw/S,2000,1,đắklắk,cntt2018,kttt,cqui,13,NaN,NaN,NaN,NaN,NaN
5082,158FB8C2XPvAibaEXe+x/3KKGWxYH2pEsBjyxoBO,1998,1,hàtĩnh,mmtt2016,mmt&tt,cqui,11,NaN,NaN,NaN,NaN,NaN
1870,1668B1AEXPvAibaEXe8LHfATqQ4bHvpZYxNsj+iv,1996,1,đắklắk,cntt0001,kttt,cqui,9,NaN,NaN,NaN,NaN,NaN
1418,3855EE8BXPvAibaEXe+ltCXpm838vada+3errwGn,1996,1,hồ chí minh,ktpm0001,cnpm,clc,9,2.0,bịcảnhbáovìđónghọcphítrễ,1.0,2019.0,22/10/2019


In [32]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
tinhtrang,6432
lydo,6432


Dữ liệu trong file 08.XLHV thể hiện việc người đó có bị cảnh cáo hay không và những dữ liệu bị khuyết sẽ được điền là 0 hoặc là không.

# Kiểm tra dữ liệu file sinhvien_dtb_toankhoa.xlsx

In [33]:
df7 = pd.read_excel(data_path + "sinhvien_dtb_toankhoa.xlsx")

In [34]:
mssv1_7 = Check_mssv(df1, df7)
mssv1_7

{'so_luong_chung': 8234, 'so_luong_khong_chung': 5797}

In [35]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df7, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,dtb_toankhoa,dtb_tichluy
7522,4C69B0DCXPvAibaEXe+3IeSFe/CqDLk+18AWDhoI,2001,1,hồ chí minh,khcl2019.2,khmt,clc,14,7.57,7.80
6270,AEB65BE3XPvAibaEXe/cAy9n2SNhGH9oyJgr5ZvZ,2000,1,hồ chí minh,mtcl2018.3,ktmt,clc,13,6.08,6.46
4260,0F864E3EXPvAibaEXe+7yjglV2dMp5y+Cm2nB6Yx,1999,1,hồ chí minh,tmđt2017,httt,cqui,12,6.52,6.81
1590,51F5CC70XPvAibaEXe8WdGcsQTF3yJIdZH0CWn7t,1996,1,quảngngãi,ktpm2014,cnpm,cqui,9,0.76,7.00
8131,F3DC1EA4XPvAibaEXe+6mZaWgr2AGUbe4cIFRSyl,2001,1,hồ chí minh,cttt2019.2,httt,cttt,14,6.59,6.64


In [36]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
dtb_toankhoa,61
dtb_tichluy,61


Dữ liệu khuyết rất ít và chỉ có ở các cột số nên sẽ xử lý bằng việc điền mean

# Kiểm tra dữ liệu file sinhvien_dtb_hocky

In [37]:
df8 = pd.read_excel(data_path + "sinhvien_dtb_hocky.xlsx")

In [38]:
mssv1_8 = Check_mssv(df1, df8)
mssv1_8

{'so_luong_chung': 8231, 'so_luong_khong_chung': 5441}

In [39]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df8, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,hocky,namhoc,dtbhk
39239,F938F519XPvAibaEXe8zDsaHJ0+9radHa8xy7Obe,1999,1,đắklắk,cntt2017,kttt,cqui,12,2.0,2018.0,7.33
46073,A0B312FEXPvAibaEXe9tVd6vhP62hwHf3WOU6N8+,2000,1,vĩnhlong,pmcl2018.1,cnpm,clc,13,1.0,2021.0,7.95
47346,FDE2D36CXPvAibaEXe8ZQB/g2cpypFECiUb9ZsQv,2000,1,lâmđồng,cntt2018,kttt,cqui,13,1.0,2018.0,7.21
8135,1A44AFDBXPvAibaEXe9Bx5CZuIA9z94nqDRZPjbc,1995,1,sóctrăng,antt2013,mmt&tt,cqui,8,1.0,2013.0,7.20
50339,1B61E745XPvAibaEXe8nuCevWXzMu5iowySn9tOD,2000,1,bìnhđịnh,mtcl2018.1,ktmt,clc,13,2.0,2021.0,7.57


In [40]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
hocky,64
namhoc,64


Dữ liệu khuyết rất ít và chỉ có ở các cột số nên sẽ xử lý bằng việc điền mean

# Kiểm tra dữ liệu file 14.totnghiep

In [41]:
df9 = pd.read_excel(data_path + "14.totnghiep.xlsx")

In [42]:
mssv1_9 = Check_mssv(df1, df9)
mssv1_9

{'so_luong_chung': 1845, 'so_luong_khong_chung': 6450}

In [43]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df9, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,xeploai
6639,7AD05EC8XPvAibaEXe/UqphXuwxfvNcMhWlmOTVr,2001,1,tiềngiang,ktpm2019,cnpm,cqui,14,NaN
113,0D5828BAXPvAibaEXe9P07hcrvmhCf8ZveiQ5xwl,1995,1,phúyên,httt2013,httt,cqui,8,NaN
4562,7CA35A03XPvAibaEXe/cJxaw79CIvjnf7zwNLUZO,1999,0,bếntre,attt2017,mmt&tt,cqui,12,NaN
321,105A1756XPvAibaEXe9KDsVv3cRkMQK+9a3JNn+S,1995,1,đồngnai,ktmt0001,ktmt,cqui,8,trungbìnhkhá
2956,E8DCBCE9XPvAibaEXe+YGniCiyMp00Njz5UXoXOM,1997,1,longan,htcl2015,httt,clc,10,NaN


In [44]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
xeploai,6450


In [45]:
merged_df.shape

(8295, 9)

# Kiểm tra dữ liệu file diemrl

In [46]:
df10 = pd.read_excel(data_path + "diemrl.xlsx")

In [47]:
mssv1_10 = Check_mssv(df1, df10)
mssv1_10

{'so_luong_chung': 8226, 'so_luong_khong_chung': 9825}

In [48]:
# Gộp hai DataFrame dựa trên các cột chung
merged_df = pd.merge(df1, df10, on=["mssv"], how="left")

# Xem kết quả
merged_df.sample(5)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,hocky,namhoc,drl,ghichu
16310,FADAFB89XPvAibaEXe88tz7NL7l/gvG6t/E6GXcF,1996,1,bìnhthuận,httt0001,httt,cqui,9,2.0,2018.0,70.0,khá
14553,5B13E979XPvAibaEXe9MSDlNDFbYdm7ZbP6bCKMi,1994,1,hảidương,khmt0001,khmt,cqui,9,2.0,2017.0,73.0,khá
1992,17946C80XPvAibaEXe9lF2o7VejWNoJR7PTzcZwM,1994,1,đànẵng,khmt2013,khmt,cqui,8,1.0,2015.0,87.0,tốt
33418,91A9AE2FXPvAibaEXe/7BYdCAEGEzTfU5A0uuq8c,1998,1,longan,httt0001,httt,cqui,11,1.0,2019.0,58.0,trungbình
57723,9EBE932BXPvAibaEXe8ZCBSJNuktiDzd68Sjee+R,2000,0,hồ chí minh,htcl2018.2,httt,clc,13,1.0,2020.0,79.0,khá


In [49]:
merged_df.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
hocky,69
namhoc,69


# Gộp dữ liệu

Các bảng sẽ xử dụng gồm có:  
01.sinhvien   
diem_THU  
0.4.xeploaiav    
0.5.ThiSinh  
0.8.XLHV  
sinhvien_dtb_toankhoa  
sinhvien_dtb_hocky

Dựa vào những kết quả trên ta thấy được là chỉ có 8234 mssv là số lượng nhỏ nhất mssv của df1 có trong các bảng vì vậy những cột nào mà số lượng nulll < 50 thì sẽ drop

Chỉnh sửa lại tên cột cho phù hợp

In [50]:
df1.columns = ['mssv', 'namsinh', 'gioitinh', 'noisinh', 'lopsh', 'khoa', 'hedt', 'khoahoc']

In [51]:
df3.columns = ['mssv', 'mamh', 'sotc', 'hocky', 'namhoc', 'diem_hp', 'trangthai']

In [52]:
df4.columns = ['mssv', 'mamta', 'diem_thita']

In [53]:
df5.columns = ['mssv', 'dien_tt', 'diem_tt', 'TEN_TRUONG']

In [54]:
df6.columns = ['mssv', 'tinhtrang', 'lydo', 'hockyxl', 'namhocxl', 'ngayqd']

In [55]:
df7.columns = ['mssv', 'dtb_toankhoa', 'dtb_tichluy']

In [56]:
df8.columns = ['mssv', 'hocky', 'namhoc', 'dtbhk']

In [57]:
df9.columns = ['mssv', 'xeploai']

In [58]:
df10.columns = ['mssv', 'hocky', 'namhoc', 'drl', 'ghichu']

Gộp dữ liệu

In [59]:
data_merge = df1.merge(df4, on='mssv', how='left')\
                .merge(df5, on='mssv', how='left')\
                .merge(df9, on='mssv', how='left')\
                .merge(df3, on='mssv', how='left')\
                .merge(df6, on='mssv', how='left')\
                .merge(df7, on='mssv', how='left')\
                .merge(df8, on=['mssv', 'hocky', 'namhoc'], how='left')\
                .merge(df10, on=['mssv', 'hocky', 'namhoc'], how='left')

In [60]:
data_merge.shape

(425822, 30)

In [61]:
data_merge.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
mamta,198220
diem_thita,198220


In [62]:
data_merge.columns

Index(['mssv', 'namsinh', 'gioitinh', 'noisinh', 'lopsh', 'khoa', 'hedt',
       'khoahoc', 'mamta', 'diem_thita', 'dien_tt', 'diem_tt', 'TEN_TRUONG',
       'xeploai', 'mamh', 'sotc', 'hocky', 'namhoc', 'diem_hp', 'trangthai',
       'tinhtrang', 'lydo', 'hockyxl', 'namhocxl', 'ngayqd', 'dtb_toankhoa',
       'dtb_tichluy', 'dtbhk', 'drl', 'ghichu'],
      dtype='object')

In [63]:
# Hàm gán giá trị vào cột ghichu dựa trên giá trị drl
def assign_rating(drl):
  '''
    Hàm gán giá trị vào cột ghichu dựa trên giá trị drl
    Param:
      drl: Giá trị có trong cột drl
  '''
  if drl >= 90:
      return 'Xuất sắc'
  elif 80 < drl <= 90:
      return 'Tốt'
  elif 65 < drl <= 80:
      return 'Khá'
  elif 50 < drl <= 65:
      return 'Trung bình'
  else:
      return 'Yếu'

In [64]:
# Xử lý dữ liệu khuyết
data_merge.dropna(subset= ['mamh', 'sotc', 'hocky', 'namhoc', 'diem_hp', 'trangthai'],inplace=True) # Drop do dữ liệu khuyết ít
data_merge['mamta'].fillna('không', inplace=True) # Điền không cho những ai không có mamta
data_merge['diem_thita'].fillna(0, inplace=True) # Điền 0 cho những ai không có mamta
data_merge['dien_tt'].fillna(data_merge['dien_tt'].mode()[0], inplace=True) # Điền theo mode
data_merge['diem_tt'].fillna(data_merge['diem_tt'].mean().round(), inplace=True) # Điền theo mean
data_merge['TEN_TRUONG'].fillna(data_merge['TEN_TRUONG'].mode()[0], inplace=True) # Điền theo mode
data_merge['xeploai'].fillna('không', inplace=True) # Điền theo không cho những ai chưa tốt nghiệp
data_merge['tinhtrang'].fillna(1, inplace=True) # Điền 1 cho những người không bị xử lý học vụ
data_merge['lydo'].fillna('không', inplace=True)  # Điền không cho những người không bị xử lý học vụ
data_merge['hockyxl'].fillna(0, inplace=True)  # Điền 0 cho những người không bị xử lý học vụ
data_merge['namhocxl'].fillna(0, inplace=True)  # Điền 0 cho những người không bị xử lý học vụ
#Coi lại cột này
data_merge['ngayqd'].fillna(0, inplace=True)  # Điền 0 cho những người không bị xử lý học vụ
data_merge['dtb_toankhoa'].fillna(data_merge['dtb_toankhoa'].mean().round(), inplace=True) # Điền theo mean
data_merge['dtb_tichluy'].fillna(data_merge['dtb_tichluy'].mean().round(), inplace=True) # Điền theo mean
data_merge['dtbhk'].fillna(data_merge['dtbhk'].mean().round(), inplace=True) # Điền theo mean
data_merge['drl'].fillna(data_merge['drl'].mean().round(), inplace=True) # Điền theo mean

<ipython-input-64-2c35c5ff43fa>:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['mamta'].fillna('không', inplace=True) # Điền không cho những ai không có mamta
<ipython-input-64-2c35c5ff43fa>:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'không' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_merge['mamta'].fillna('không', inplace=True) # Điền không cho những ai không có mamta
<ipytho

In [65]:
# Cập nhật cột ghichu
data_merge['ghichu'] = data_merge['drl'].apply(assign_rating)

# Xóa khoảng trắng thừa trong cột "ghichu"
data_merge['ghichu'] = data_merge['ghichu'].str.strip()  # Xóa khoảng trắng đầu và cuối
data_merge['ghichu'] = data_merge['ghichu'].str.replace(r'\s+', '', regex=True)  # Xóa khoảng trắng thừa giữa các từ

In [66]:
data_merge.shape

(425800, 30)

In [67]:
data_merge.isnull().sum()

,0
mssv,0
namsinh,0
gioitinh,0
noisinh,0
lopsh,0
khoa,0
hedt,0
khoahoc,0
mamta,0
diem_thita,0


In [68]:
data_merge.info()

<class 'pandas.core.frame.DataFrame'>
Index: 425800 entries, 0 to 425821
Data columns (total 30 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   mssv          425800 non-null  object 
 1   namsinh       425800 non-null  int64  
 2   gioitinh      425800 non-null  int64  
 3   noisinh       425800 non-null  object 
 4   lopsh         425800 non-null  object 
 5   khoa          425800 non-null  object 
 6   hedt          425800 non-null  object 
 7   khoahoc       425800 non-null  int64  
 8   mamta         425800 non-null  object 
 9   diem_thita    425800 non-null  object 
 10  dien_tt       425800 non-null  object 
 11  diem_tt       425800 non-null  float64
 12  TEN_TRUONG    425800 non-null  object 
 13  xeploai       425800 non-null  object 
 14  mamh          425800 non-null  object 
 15  sotc          425800 non-null  float64
 16  hocky         425800 non-null  float64
 17  namhoc        425800 non-null  float64
 18  diem_hp  

In [69]:
data_merge.dtypes

,0
mssv,object
namsinh,int64
gioitinh,int64
noisinh,object
lopsh,object
khoa,object
hedt,object
khoahoc,int64
mamta,object
diem_thita,object


Tạo thêm cột namnhaphoc để xác định năm nhập học của sinh viên

In [70]:
# Lấy dữ liệu về khóa học của sinh viên
khoaHoc = data_merge['khoahoc'].copy()

lis = np.unique(khoaHoc) # Tạo danh sách các khóa học có trong bảng dữ liệu
lis_replace = np.array([2013, 2014, 2015, 2016, 2017, 2018, 2019])  # Tạo danh sách các năm học tương ứng với các khóa
khoaHoc.replace(to_replace = lis, value = lis_replace, inplace = True) # Thay thế các giá trị khóa học trong biến khoaHoc thành namnhaphoc
data_merge['namnhaphoc'] = khoaHoc

Tạo thêm cột sinhvien_nam để xác định sinh siên hiện tại là sinh viên năm mấy

In [71]:
data_merge['sinhvien_nam'] = data_merge['namhoc'] - data_merge['namnhaphoc'] + 1 # Tính hiện tại sinh viên là sinh viên năm mấy

In [72]:
data_merge = data_merge.drop(columns=['namnhaphoc'])

In [73]:
data_merge.sample(200)

,mssv,namsinh,gioitinh,noisinh,lopsh,khoa,hedt,khoahoc,mamta,diem_thita,...,lydo,hockyxl,namhocxl,ngayqd,dtb_toankhoa,dtb_tichluy,dtbhk,drl,ghichu,sinhvien_nam
205252,FE3AC989XPvAibaEXe8VnqU11737Hl6IFmKBfB+P,1997,0,nghệan,mmtt0001,mmt&tt,cqui,11,20.0,avsc,...,không,0.0,0.0,0,6.90,6.96,6.04,83.0,Tốt,1.0
179310,66A9B4FBXPvAibaEXe+BUGeEgVnB4P/PVf37xsAo,1997,1,hồ chí minh,httt0001,httt,cqui,10,không,0,...,bịcảnhbáovìđtbhọckỳ2,1.0,2019.0,22/10/2019,5.33,6.65,6.53,81.0,Tốt,3.0
187432,81591236XPvAibaEXe9a9WHB728zIC+BsRhdRUah,1997,1,nghệan,mmtt0001,mmt&tt,cqui,11,34.0,avsc,...,bịcảnhbáovìđónghọcphítrễ,2.0,2019.0,19/05/2020,6.76,6.81,6.11,80.0,Khá,1.0
33305,99325C17XPvAibaEXe8wlx2/dthDAuX2r9IBWF/q,1995,1,đồngnai,khmt2013,khmt,cqui,8,không,0,...,không,0.0,0.0,0,7.37,7.37,5.98,59.0,Trungbình,1.0
222197,2A1CE9C9XPvAibaEXe86ShnmDByrmwlM6+Yi6Z7F,1998,1,hồ chí minh,ktmt0001,ktmt,clc,11,57.0,en002,...,không,0.0,0.0,0,7.13,7.30,7.23,80.0,Khá,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121944,8DB5C6EAXPvAibaEXe8WA4OgPtanSMEjuq/KlO99,1996,1,tiềngiang,ktpm0001,cnpm,clc,9,không,0,...,không,0.0,0.0,0,7.40,7.61,7.89,86.0,Tốt,1.0
383544,7B7F993CXPvAibaEXe++B0JyCbXA1DLReLiuwrPr,2001,0,đồngnai,cncl2019.2,kttt,clc,14,không,0,...,không,0.0,0.0,0,7.52,7.52,8.39,100.0,Xuấtsắc,3.0
381039,82F358D5XPvAibaEXe+Pxorfe/k8CeQSGF32nC9q,2001,1,hồ chí minh,mmcl2019.1,mmt&tt,clc,14,365.0,toeic,...,bịcảnhbáovìđiểmtrungbình2họckỳliêntiếp<4,2.0,2020.0,28/03/2019,4.05,6.36,3.67,75.0,Khá,3.0
351032,E0EA07F4XPvAibaEXe8qwxsYP7Gy4egolsydGMpR,2000,1,khánhhòa,mmtt2018,mmt&tt,cqui,13,66.0,eng01,...,buộcthôihọcđượchộiđồngxemxéthạmức,2.0,2020.0,28/03/2019,4.81,6.40,4.84,63.0,Trungbình,2.0


In [75]:
# Kiểm tra các giá trị có trong thuộc tính sinhvien_nam
data_merge['sinhvien_nam'].unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9.,  0., 10.])

In [76]:
# Loại bỏ các hàng có sinhvien_nam = 0
data_merge = data_merge[data_merge['sinhvien_nam'] != 0]

In [77]:
# Kiểm tra các giá trị có trong thuộc tính sinhvien_nam
data_merge['sinhvien_nam'].unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])

In [78]:
data_merge.to_excel('/content/drive/MyDrive/IS353 - Mạng xã hội - Nhóm 6/Đồ án môn học/Project/Data/Data_merge/data_merge.xlsx', index=False)